In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

SRC  = "supplychainiq.bronze.inventory"
TGT  = "supplychainiq.silver.inventory"
QUAR = "supplychainiq.silver.inventory_quarantine"
CKPT = "/Volumes/supplychainiq/ops/checkpoints/silver_inventory"
LOOKBACK, MIN_HISTORY, SPIKE_FACTOR, SPIKE_PAD = 28, 7, 5, 100
KEYS = ["snapshot_date", "store_id", "item_id"]

spark.sql(f"""CREATE TABLE IF NOT EXISTS {TGT} (
  snapshot_date DATE, store_id STRING, item_id STRING, on_hand INT,
  _source_file STRING, _ingested_at TIMESTAMP, _silver_at TIMESTAMP)
  CLUSTER BY (snapshot_date, store_id)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {QUAR} (
  snapshot_date DATE, store_id STRING, item_id STRING, on_hand INT,
  prev_max_28d INT, prev_days BIGINT, _rescued_data STRING,
  _source_file STRING, _ingested_at TIMESTAMP, _dq_reason STRING, _quarantined_at TIMESTAMP)""")

In [0]:
def process(batch_df, batch_id):
    s = batch_df.sparkSession
    new = batch_df.select(
        F.to_date("snapshot_date").alias("snapshot_date"), "store_id", "item_id",
        F.col("on_hand").cast("int").alias("on_hand"),
        "_rescued_data", "_source_file", "_ingested_at")
    rng = new.agg(F.min("snapshot_date").alias("lo"), F.max("snapshot_date").alias("hi")).first()
    if rng["lo"] is None:
        return

    # history = already-clean Silver rows around this batch (bad values never enter it)
    hist = (s.table(TGT)
              .where(F.col("snapshot_date").between(F.date_sub(F.lit(rng["lo"]), LOOKBACK), F.lit(rng["hi"])))
              .select("snapshot_date", "store_id", "item_id", "on_hand",
                      F.lit(None).cast("string").alias("_rescued_data"),
                      "_source_file", "_ingested_at"))
    new = new.join(hist.select(*KEYS), KEYS, "left_anti")        # retried batch -> skip rows already in Silver

    w = Window.partitionBy("store_id", "item_id").orderBy("snapshot_date").rowsBetween(-LOOKBACK, -1)
    scored = (new.withColumn("_is_new", F.lit(True))
                .unionByName(hist.withColumn("_is_new", F.lit(False)))
                .withColumn("prev_max_28d", F.max("on_hand").over(w))
                .withColumn("prev_days", F.count("on_hand").over(w))
                .filter("_is_new"))

    reason = (F.when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("on_hand").isNull() | F.col("snapshot_date").isNull(), "missing_value")
               .when(F.col("on_hand") < 0, "negative_inventory")
               .when((F.col("prev_days") >= MIN_HISTORY) &
                     (F.col("on_hand") > F.col("prev_max_28d") * SPIKE_FACTOR + SPIKE_PAD),
                     "impossible_inventory"))
    scored = scored.withColumn("_dq_reason", reason)

    (scored.filter("_dq_reason IS NOT NULL")
        .withColumn("_quarantined_at", F.current_timestamp())
        .select(*s.table(QUAR).columns)
        .write.mode("append")
        .option("txnAppId", "silver_inventory_quarantine").option("txnVersion", batch_id)
        .saveAsTable(QUAR))

    good = (scored.filter("_dq_reason IS NULL")
              .withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), " AND ".join(f"t.{k} = s.{k}" for k in KEYS))
        .whenNotMatchedInsertAll()
        .execute())

before = spark.table(TGT).count()
q = (spark.readStream
       .option("maxBytesPerTrigger", "64m")       # small micro-batches: avoids the session crash
       .table(SRC)
       .writeStream.foreachBatch(process)
       .option("checkpointLocation", CKPT)
       .trigger(availableNow=True)
       .start())
q.awaitTermination()
after = spark.table(TGT).count()
print(f"{after - before:,} clean snapshots added (total {after:,}); "
      f"quarantine total {spark.table(QUAR).count():,}")

In [0]:
%sql
WITH truth AS (
  SELECT to_date(date) AS d, store_id, item_id, fault_type
  FROM supplychainiq.ops.fault_truth
  WHERE fault_type IN ('negative_inventory', 'impossible_inventory') AND to_date(date) <= '2015-08-20'),
flagged AS (
  SELECT snapshot_date AS d, store_id, item_id, _dq_reason FROM supplychainiq.silver.inventory_quarantine)
SELECT
  (SELECT COUNT(*) FROM supplychainiq.bronze.inventory)                       AS bronze_rows,
  (SELECT COUNT(*) FROM supplychainiq.silver.inventory)                       AS silver_rows,
  (SELECT COUNT(*) FROM flagged)                                              AS quarantined,
  (SELECT COUNT(*) FROM truth)                                                AS truth_bad,
  (SELECT COUNT(*) FROM truth JOIN flagged USING (d, store_id, item_id))      AS caught,
  (SELECT COUNT(*) FROM flagged LEFT ANTI JOIN truth USING (d, store_id, item_id)) AS false_alarms;

In [0]:
%sql
SELECT q.* FROM supplychainiq.silver.inventory_quarantine q
LEFT ANTI JOIN (SELECT to_date(date) AS snapshot_date, store_id, item_id FROM supplychainiq.ops.fault_truth
                WHERE fault_type LIKE '%inventory') t USING (snapshot_date, store_id, item_id)
ORDER BY on_hand DESC LIMIT 20;